In [ ]:
from PIL import Image
import numpy as np
import cv2
mask_path = "Semantic segmentation dataset\Tile 1\masks\image_part_001.png"
img_path = 
mask = np.array(Image.open(mask_path).convert("RGB"))

colors = np.unique(mask.reshape(-1, 3), axis=0)

print(colors)
print("Liczba kolorów:", len(colors))

[[ 60  16 152]
 [110 193 228]
 [132  41 246]
 [155 155 155]
 [226 169  41]
 [254 221  58]]
Liczba kolorów: 6


In [15]:

mask = cv2.imread(mask_path)
mask = cv2.cvtColor(mask,cv2.COLOR_BGR2RGB)
encoded_mask = np.zeros((mask.shape[0],mask.shape[1]), dtype=np.uint8)

encoded_mask[(mask == (132, 41, 246)).all(axis=-1)] = 1
print(encoded_mask)


[[1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 ...
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]
 [1 1 1 ... 1 1 1]]


In [24]:
def mask_encoding(mask):
    mask = cv2.cvtColor(mask,cv2.COLOR_BGR2RGB)
    colors_map = {
        (60, 16, 152): 0, # Building
        (132, 41, 246): 1,# Land
        (110, 193, 228): 2,# Road
        (254, 221, 58): 3,# Vegetation
        (226, 169, 41): 4,# Water
        (155, 155, 155): 5 # Unlabeled
    }
    encoded_mask = np.zeros((mask.shape[0],mask.shape[1]), dtype=np.uint8)
    for color, idx in colors_map.items():
        encoded_mask[(mask == color).all(axis=-1)] = idx
    return encoded_mask


In [60]:
from pathlib import Path
import matplotlib.pyplot as plt
def cut_into_patches(img_path, mask_path,out_img_dir, out_mask_dir, patch_size=256):
    

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    
    mask = cv2.imread(mask_path)
    mask = mask_encoding(mask)

    out_img_dir = Path(out_img_dir)
    out_mask_dir = Path(out_mask_dir)
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_mask_dir.mkdir(parents=True, exist_ok=True)

    base_name = Path(img_path).stem
    # plt.imshow(img)
    # plt.show()
    h,w,_ = img.shape
    cntr = 0
    for y in range(0,h-patch_size+1,patch_size):
        for x in range(0,w-patch_size+1,patch_size):
            cropped_img = np.array(img[y:y+patch_size,x:x+patch_size],dtype=np.uint8)
            cropped_mask = np.array(mask[y:y+patch_size,x:x+patch_size],dtype=np.uint8)
            # plt.imshow(cropped_img)
            # plt.show()
            img_patch_path = out_img_dir / f"{base_name}_{cntr}.png"
            mask_patch_path = out_mask_dir / f"{base_name}_{cntr}.png"
            cv2.imwrite(img_patch_path,cropped_img)
            cv2.imwrite(mask_patch_path,cropped_mask)
            cntr+=1


In [63]:
from pathlib import Path
patch_size = 1024
RAW_DATA_DIR = Path("Semantic segmentation dataset")
OUT_DIR = Path("prepared_data"+f"{patch_size}")

OUT_IMAGES = OUT_DIR / "images"
OUT_MASKS = OUT_DIR / "masks"



for tile_dir in RAW_DATA_DIR.glob("Tile *"):
    images_dir = tile_dir / "images"
    masks_dir = tile_dir / "masks"

    if not images_dir.exists() or not masks_dir.exists():
        continue

    image_paths = sorted(images_dir.glob("*.jpg")) + sorted(images_dir.glob("*.png"))
    # print(image_paths)
    # img=cv2.imread(image_paths[0])
    # plt.imshow(img)
    # plt.show()
    for image_path in image_paths:
        mask_path = (masks_dir / image_path.stem).with_suffix('.png')
        # print(mask_path)
        # img=cv2.imread(mask_path)
        # plt.imshow(img)
        # plt.show()
        if not mask_path.exists():
            print("Brak maski dla:", image_path)
            continue

        cut_into_patches(
            img_path=image_path,
            mask_path=mask_path,
            out_img_dir=OUT_IMAGES,
            out_mask_dir=OUT_MASKS,
            patch_size=patch_size
        )

In [ ]:
m=cv2.imread("prepared_data256\masks\image_part_001_3.png")


In [ ]:

images =